# SOTA Championship V3: High-Compute Retraining & Surveillance Adaptation
### WACV 2027 UPAR Challenge Track 2 | Resume Training from best_model.pt | Target: mADM >= 0.7500+
---
This notebook executes **Phase 2 Resumed Retraining** of the SOTA Championship V3 architecture on Kaggle Dual Tesla T4 GPUs.

### Why Resume Training from best_model.pt?
1. **Pre-converged Foundations:** best_model.pt already achieved 0.5720 mADM on the CodaBench server, establishing strong metric alignment.
2. **Anatomical & Deformable Convergence:** Continuing for 30 epochs with lower learning rates (1e-5 trunk, 8e-5 surgical) allows the 6 AGCA part tokens and DMHA offsets to fully specialize.
3. **Immediate Surveillance Degradation (SRDM++):** Injects heavy resolution drops (down to 48px), motion blur, and sensor noise from epoch 1 to bridge the gap to MEVID surveillance images.
4. **Full Compute Utilization:** Packages the retrained model with 3-Pass TTA and the 1,500-candidate test-time compute suite.


In [ ]:
# ==============================================================================
# CELL 1: HARDWARE & SYSTEM TELEMETRY CHECK
# ==============================================================================
import os
import sys
import psutil
import subprocess
import torch

print("=" * 80)
print("[HARDWARE] Inspecting Dual Tesla T4 Accelerators & Host Environment...")
print("=" * 80)

cuda_available = torch.cuda.is_available()
gpu_count = torch.cuda.device_count() if cuda_available else 0
print(f"CUDA Available: {cuda_available} | Device Count: {gpu_count}")

for i in range(gpu_count):
    props = torch.cuda.get_device_properties(i)
    print(f"  -> GPU {i}: {props.name} | Total VRAM: {props.total_memory / (1024**3):.2f} GB")

vmem = psutil.virtual_memory()
print(f"Host CPU Cores: {os.cpu_count()} | Total RAM: {vmem.total / (1024**3):.2f} GB | Available: {vmem.available / (1024**3):.2f} GB")
print("=" * 80)

# Install required SOTA vision-language dependencies
print("[DEPS] Installing OpenCLIP, Timm, and Accelerate dependencies...")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "open-clip-torch", "timm", "accelerate"], check=True)
print("[DEPS] Dependencies installed successfully!")


In [ ]:
# ==============================================================================
# CELL 2: CLONE OR PULL SOTA V3 REPOSITORY VIA GITHUB_TOKEN
# ==============================================================================
import os
import sys
import subprocess

REPO_NAME = "upar_challenge_26_rws27"
REPO_DIR = os.path.join("/kaggle/working", REPO_NAME)

token = None
try:
    from kaggle_secrets import UserSecretsClient
    user_secrets = UserSecretsClient()
    token = user_secrets.get_secret("GITHUB_TOKEN")
    print("[AUTH] Successfully retrieved GITHUB_TOKEN from Kaggle Secrets!")
except Exception as e:
    print(f"[AUTH] Note: Kaggle Secrets not accessible ({e}). Attempting public clone...")

if token:
    repo_url = f"https://{token}@github.com/Aditya-Lingam-9000/{REPO_NAME}.git"
else:
    repo_url = f"https://github.com/Aditya-Lingam-9000/{REPO_NAME}.git"

if not os.path.isdir(REPO_DIR):
    print(f"[GIT] Cloning repository into {REPO_DIR}...")
    res = subprocess.run(["git", "clone", repo_url, REPO_DIR], capture_output=True, text=True)
    if res.returncode != 0:
        print(f"[GIT ERROR] Clone failed:\n{res.stderr}")
        raise RuntimeError("Failed to clone repository. Verify GITHUB_TOKEN.")
    print("[GIT] Successfully cloned repository!")
else:
    print(f"[GIT] Repository already exists at {REPO_DIR}. Fetching latest commits from main...")
    res = subprocess.run(["git", "-C", REPO_DIR, "pull", "origin", "main"], capture_output=True, text=True)
    print(res.stdout)

if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
os.chdir(REPO_DIR)
print(f"[SYSTEM] Working directory set to: {os.getcwd()}")


In [ ]:
# ==============================================================================
# CELL 3: MASTER SOTA V3 RESUME RETRAINING CONFIGURATION
# ==============================================================================
import os
import sys
from sota_pipeline.config import SOTAConfig

# Resume Retraining Configuration (Fine-Tuning from best_model.pt)
EPOCHS = 30                  # Additional fine-tuning epochs with active surveillance degradations
BATCH_SIZE_PER_GPU = 128     # 128 per GPU * 2 GPUs = Global: 256 (VRAM: ~13.5 GB / GPU)
ACCUMULATION_STEPS = 2       # Effective global batch size = 512
TRUNK_LR = 1.0e-5            # Refined trunk learning rate (gentle backbone fine-tuning)
SURGICAL_LR = 8.0e-5         # Refined surgical learning rate for AGCA part tokens & prompter
MIN_LR = 5e-7                # Learning rate floor
INITIAL_CURRICULUM = 0.60    # Start CBS curriculum at 0.60 (immediate severe surveillance blur & downsampling)
EVAL_EVERY = 1               # Run validation retrieval benchmark after EVERY epoch
VAL_QUERIES = 500            # Number of validation queries for exact CodaBench benchmarking
EVAL_BATCH_SIZE = 384        # Fast validation batch size (<25s per evaluation)
CHECKPOINT_DIR = "checkpoints_sota_v3_resumed"
OUTPUT_DIR = "sota_v3_resumed_outputs"

# Checkpoint Resume Path resolution
candidate_checkpoints = [
    "best_model.pt",
    "/kaggle/input/best-model/best_model.pt",
    "/kaggle/working/best_model.pt",
    "sota_v3_outputs/best_model_sota_v3.pt",
    "notebooks/best_model.pt",
]
RESUME_FROM = None
for cand in candidate_checkpoints:
    if os.path.isfile(cand):
        RESUME_FROM = cand
        break

DATASET_ROOT = SOTAConfig.resolve_dataset_root()

print("=" * 80)
print(f"[CONFIG] SOTA V3 RESUME RETRAINING CONFIGURATION:")
print(f"  - Resume Checkpoint: {RESUME_FROM}")
print(f"  - Additional Epochs: {EPOCHS}")
print(f"  - Initial CBS Surveillance Curriculum: {INITIAL_CURRICULUM} (Severe degradations from epoch 1)")
print(f"  - Batch Size: {BATCH_SIZE_PER_GPU}/GPU (Global: {BATCH_SIZE_PER_GPU * 2}, Effective: {BATCH_SIZE_PER_GPU * 2 * ACCUMULATION_STEPS})")
print(f"  - Trunk LR: {TRUNK_LR:.2e} | Surgical LR: {SURGICAL_LR:.2e}")
print(f"  - Dataset Root: {DATASET_ROOT}")
print("=" * 80)


In [ ]:
# ==============================================================================
# CELL 4: HUGGINGFACE ACCELERATE DUAL T4 MULTI-GPU CONFIGURATION
# ==============================================================================
import os
import yaml

accelerate_config = {
    "compute_environment": "LOCAL_MACHINE",
    "distributed_type": "MULTI_GPU",
    "num_machines": 1,
    "num_processes": 2,
    "gpu_ids": "all",
    "mixed_precision": "fp16",
    "main_process_port": 29500,
    "rdzv_backend": "static",
    "same_network": True,
}

config_path = "accelerate_config.yaml"
with open(config_path, "w") as f:
    yaml.dump(accelerate_config, f, default_flow_style=False)

print(f"[ACCELERATE] Generated multi-GPU configuration at: {os.path.abspath(config_path)}")


In [ ]:
# ==============================================================================
# CELL 5: LAUNCH SOTA V3 RESUME TRAINING WITH ACCELERATE
# ==============================================================================
import os
import sys
import shutil
import subprocess

if shutil.which("accelerate"):
    launcher = ["accelerate", "launch", "--config_file", "accelerate_config.yaml"]
else:
    launcher = ["torchrun", "--nproc_per_node=2"]

cmd = launcher + [
    "-m", "sota_pipeline.train_v3",
    f"--epochs={EPOCHS}",
    f"--batch_size={BATCH_SIZE_PER_GPU}",
    f"--accumulation_steps={ACCUMULATION_STEPS}",
    f"--trunk_lr={TRUNK_LR}",
    f"--surgical_lr={SURGICAL_LR}",
    f"--dataset_root={DATASET_ROOT}",
    f"--checkpoint_dir={CHECKPOINT_DIR}",
    f"--output_dir={OUTPUT_DIR}",
    f"--eval_every={EVAL_EVERY}",
    f"--initial_curriculum_progress={INITIAL_CURRICULUM}",
]
if RESUME_FROM:
    cmd.append(f"--resume_from={RESUME_FROM}")

print("[EXEC] Executing SOTA V3 Resume Retraining Command:")
print(" ".join(cmd))
print("=" * 80)
subprocess.run(cmd, check=True)


In [ ]:
# ==============================================================================
# CELL 6: POST-TRAINING CHECKPOINT & SWAD INTEGRITY VERIFICATION
# ==============================================================================
import os
import torch

print("=" * 80)
print("[VERIFY] Auditing Saved SOTA V3 Checkpoints & SWAD Basin Weights...")
print("=" * 80)

best_path = os.path.join(OUTPUT_DIR, "best_model_sota_v3.pt")
swad_path = os.path.join(OUTPUT_DIR, "model_sota_v3_swad_fp16.pt")

if os.path.isfile(best_path):
    ckpt = torch.load(best_path, map_location="cpu", weights_only=False)
    metrics = ckpt.get("metrics", {})
    epoch = ckpt.get("epoch", "Unknown")
    print(f"Best Checkpoint: {best_path} ({os.path.getsize(best_path) / (1024**2):.2f} MB)")
    print(f"  - Saved at Epoch: {epoch}")
    print(f"  - Best Validation Metrics:")
    print(f"      * mADM:   {metrics.get('mADM', 0.0):.4f} (Live #1: 0.6783)")
    print(f"      * Rank-1: {metrics.get('Rank-1', 0.0):.2f}% (Live #1: 40.79%)")
    print(f"      * Rank-5: {metrics.get('Rank-5', 0.0):.2f}%")
    print(f"      * mAP:    {metrics.get('mAP', 0.0):.2f}%")

if os.path.isfile(swad_path):
    swad_mb = os.path.getsize(swad_path) / (1024**2)
    print(f"\nSWAD Centroid Model: {swad_path} ({swad_mb:.2f} MB)")
    print("  -> Dense basin centroid ready for ultra-compact deployment!")
print("=" * 80)


In [ ]:
# ==============================================================================
# CELL 7: AUTOMATED SOTA V3 RESUMED SUBMISSION ZIP PACKAGING
# ==============================================================================
import os
import sys
import subprocess

ZIP_DEST = "/kaggle/working/submission_sota_v3_resumed.zip"
CHECKPOINT_PATH = os.path.join(OUTPUT_DIR, "best_model_sota_v3.pt")
TEMPLATE_DIR = "sota_pipeline/submission_template_v3"

print("=" * 80)
print("[PACKAGING] Exporting Retrained Model & Generating CodaBench ZIP...")
print("=" * 80)

export_cmd = [
    sys.executable, "sota_pipeline/export_submission_v3.py",
    f"--checkpoint={CHECKPOINT_PATH}",
    f"--template_dir={TEMPLATE_DIR}",
    f"--output_zip={ZIP_DEST}",
]
subprocess.run(export_cmd, check=True)

zip_size_mb = os.path.getsize(ZIP_DEST) / (1024 * 1024)
print(f"
[PACKAGE COMPLETE] Generated: {ZIP_DEST} ({zip_size_mb:.2f} MB)")
print("=" * 80)


In [ ]:
# ==============================================================================
# CELL 8: ISOLATED SANDBOX CONTAINER STRUCTURAL AUDIT & RUNTIME TEST
# ==============================================================================
import os
import sys
import ast
import shutil
ZIP_PATH = "/kaggle/working/submission_sota_v3_resumed.zip"
import tempfile
import subprocess
import torch

ZIP_PATH = "/kaggle/working/submission_sota_v3.zip"
assert os.path.isfile(ZIP_PATH), f"ZIP not found at {ZIP_PATH}!"

SANDBOX_DIR = tempfile.mkdtemp(prefix="codabench_v3_sandbox_audit_")
try:
    print("=" * 80)
    print(f"[SANDBOX AUDIT] Extracting {ZIP_PATH} into clean sandbox: {SANDBOX_DIR}...")
    with zipfile.ZipFile(ZIP_PATH, "r") as zf:
        zf.extractall(SANDBOX_DIR)

    audit_checks = []

    # 1. run.py existence
    r_path = os.path.join(SANDBOX_DIR, "run.py")
    c1 = os.path.isfile(r_path) and os.path.getsize(r_path) > 10000
    audit_checks.append(("run.py existence and integrity", c1, f"{os.path.getsize(r_path):,} bytes"))

    # 2. metadata.yaml existence
    m_path = os.path.join(SANDBOX_DIR, "metadata.yaml")
    c2 = os.path.isfile(m_path)
    audit_checks.append(("metadata.yaml descriptor presence", c2, "Found"))

    # 3. Model weights existence & size
    w_path = os.path.join(SANDBOX_DIR, "assets", "model_sota_fp16.pt")
    w_mb = os.path.getsize(w_path) / (1024**2) if os.path.isfile(w_path) else 0
    c3 = os.path.isfile(w_path) and w_mb > 200
    audit_checks.append(("model_sota_fp16.pt (> 200 MB)", c3, f"{w_mb:.2f} MB"))

    # 4. Vendored timm module for zero-internet execution
    v_path = os.path.join(SANDBOX_DIR, "vendor", "timm")
    c4 = os.path.isdir(v_path) and os.path.isfile(os.path.join(v_path, "__init__.py"))
    audit_checks.append(("vendor/timm air-gapped module", c4, "Directory verified"))

    # 5. AST syntax parse & CodaBench entrypoints check
    with open(r_path, "r", encoding="utf-8") as f:
        code_text = f.read()
    parsed_tree = ast.parse(code_text)
    func_names = [n.name for n in ast.walk(parsed_tree) if isinstance(n, ast.FunctionDef)]
    c5 = all(fn in func_names for fn in ["load_model", "rank_gallery"])
    audit_checks.append(("CodaBench entrypoints (load_model, rank_gallery)", c5, "Both confirmed"))

    # 6. Air-gapped code hygiene (no hardcoded absolute machine paths)
    c6 = ("/kaggle/" not in code_text) and ("/Users/" not in code_text) and ("/home/" not in code_text)
    audit_checks.append(("Air-gapped code hygiene (no hardcoded host paths)", c6, "Clean"))

    # 7. Structural weight dictionary inspection
    sd = torch.load(w_path, map_location="cpu", weights_only=False)
    has_vis = any(k.startswith("visual_stream.") for k in sd) or any(k.startswith("trunk.") for k in sd)
    all_fp16 = all(v.dtype == torch.float16 for v in sd.values() if isinstance(v, torch.Tensor) and v.is_floating_point())
    has_nan = any(torch.isnan(v).any().item() for v in sd.values() if isinstance(v, torch.Tensor))
    has_inf = any(torch.isinf(v).any().item() for v in sd.values() if isinstance(v, torch.Tensor))
    c7 = has_vis and (not has_nan) and (not has_inf) and all_fp16
    audit_checks.append(("FP16 weight tensor validity (0 NaNs, 0 Infs)", c7, f"FP16: {all_fp16}"))

    # 8. Archive size compliance
    zip_mb = os.path.getsize(ZIP_PATH) / (1024 * 1024)
    c8 = zip_mb < 500.0
    audit_checks.append(("Archive size compliance (< 500 MB)", c8, f"{zip_mb:.2f} MB"))

    # 9. Test-Time Compute Engine integration check
    c9 = "TestTimeComputeEngine" in code_text and "run_inference_pipeline" in code_text
    audit_checks.append(("Test-Time Compute Engine integrated in run.py", c9, "Confirmed"))

    # 10. Air-gapped runtime execution test inside sandbox
    res = subprocess.run([sys.executable, r_path], cwd=SANDBOX_DIR, capture_output=True, text=True)
    c10 = res.returncode == 0 and "Self-test SUCCESS!" in res.stdout
    audit_checks.append(("In-situ air-gapped test execution", c10, "Self-test SUCCESS!"))

    print("=" * 80)
    print("ISOLATED SANDBOX CONTAINER STRUCTURAL AUDIT SCORECARD")
    print("=" * 80)
    all_passed = True
    for name, passed, detail in audit_checks:
        status = "[PASS]" if passed else "[FAIL]"
        print(f"  {status:7s} | {name:<50s} | {detail}")
        if not passed:
            all_passed = False
    print("=" * 80)
    if all_passed:
        print("[AUDIT VERDICT] ALL 10/10 CHECKS PASSED WITH FLYING COLORS!")
        print("  -> Package is 100% self-contained, air-gapped, and competition-ready.")
        print(f"  -> Download '{ZIP_PATH}' and submit to CodaBench!")
    else:
        print("[AUDIT VERDICT] AUDIT FAILED! Review failing checks above.")
        raise RuntimeError("Structural audit failed! Do not submit this archive.")
    print("=" * 80)
finally:
    if os.path.exists(SANDBOX_DIR):
        shutil.rmtree(SANDBOX_DIR)
        print(f"[CLEANUP] Successfully removed sandbox directory: {SANDBOX_DIR}")
